# Frame Type Analyzer

Analyzes an existing video file's I/P/B frame distribution, GOP size, and motion vector density via `ffprobe`, to help find the most 'corruptible' segments before running `tools/ffmpeg_datamosh.py` on it.

Requires `ffprobe` on PATH.

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

VIDEO_PATH = Path('../sample.mp4')  # point this at a real file to analyze

assert shutil.which('ffprobe'), "ffprobe not found on PATH"

## 1. Pull per-frame metadata

`pict_type` gives us I/P/B classification. `pkt_size` (bitstream size of the frame) is a decent proxy for 'how much residual/motion information this frame carried' — I-frames are typically much larger.

In [ ]:
def probe_frames(path: Path):
    out = subprocess.run([
        'ffprobe', '-v', 'quiet', '-select_streams', 'v:0',
        '-show_entries', 'frame=pict_type,pkt_size,pkt_pts_time',
        '-of', 'json', str(path),
    ], capture_output=True, text=True, check=True)
    return json.loads(out.stdout)['frames']

frames = probe_frames(VIDEO_PATH)
print(f'Total frames: {len(frames)}')
frames[:5]

## 2. Frame type distribution

In [ ]:
types = [f.get('pict_type', '?') for f in frames]
unique, counts = np.unique(types, return_counts=True)

plt.bar(unique, counts)
plt.title('Frame type distribution')
plt.xlabel('Frame type')
plt.ylabel('Count')
for t, c in zip(unique, counts):
    print(f'{t}: {c} ({c / len(types):.1%})')

## 3. GOP size over time

Distance (in frames) between consecutive I-frames. Smaller GOPs mean more frequent 'safe' resync points; larger GOPs mean longer runs of P/B-frames that can be exploited for a longer, uninterrupted smear.

In [ ]:
i_frame_indices = [i for i, t in enumerate(types) if t == 'I']
gop_sizes = np.diff(i_frame_indices) if len(i_frame_indices) > 1 else np.array([])

print(f'I-frame positions: {i_frame_indices}')
print(f'GOP sizes: {gop_sizes}')
if len(gop_sizes):
    print(f'Mean GOP size: {gop_sizes.mean():.1f} frames')
    plt.figure()
    plt.plot(gop_sizes, marker='o')
    plt.title('GOP size over time')
    plt.xlabel('GOP index')
    plt.ylabel('Frames until next I-frame')

## 4. Bitstream size per frame (residual/motion proxy)

Large `pkt_size` spikes on P/B-frames usually mean high motion or a scene change the encoder struggled to predict — these are good candidates for dramatic datamosh corruption, since the decoder is already leaning hard on the (removable) reference.

In [ ]:
sizes = np.array([int(f.get('pkt_size', 0)) for f in frames])
colors = {'I': 'red', 'P': 'blue', 'B': 'green'}

plt.figure(figsize=(12, 4))
for t in set(types):
    idx = [i for i, tt in enumerate(types) if tt == t]
    plt.scatter(idx, sizes[idx], s=8, label=t, color=colors.get(t, 'gray'))
plt.legend()
plt.title('Packet size per frame')
plt.xlabel('Frame index')
plt.ylabel('Packet size (bytes)')

## 5. Suggest corruption candidates

Heuristic: flag GOPs that are both long (lots of P/B-frames to corrupt) and contain a high-motion spike (likely to produce dramatic, visible smear when the I-frame anchoring them is removed).

In [ ]:
def suggest_candidates(i_frame_indices, sizes, types, top_n=5):
    candidates = []
    bounds = list(zip(i_frame_indices, i_frame_indices[1:] + [len(types)]))
    for start, end in bounds:
        gop_len = end - start
        gop_sizes = sizes[start:end]
        score = gop_len * gop_sizes.std()
        candidates.append((start, end, gop_len, score))
    candidates.sort(key=lambda c: c[3], reverse=True)
    return candidates[:top_n]

for start, end, length, score in suggest_candidates(i_frame_indices, sizes, types):
    print(f'Frames {start}-{end} (len {length}): score={score:.0f}')